# 🤖 SmolVLA GPU 訓練（黑盒子飛行記錄器 / 斷線自動續訓版）

**為什麼 Colab 會莫名中斷？**
1. 💥 **系統 RAM 耗盡 (Linux OOM Killer)**：4 個 DataLoader worker 與圖像快取吃光 12.7 GB RAM，Linux 會直接 `kill -9` 強制終止程序，**完全不留 Python 錯誤訊息**。
2. 💾 **本機磁碟爆滿 (Disk Full)**：每個 Checkpoint 數 GB，若累積過多，磁碟達到 100% 系統會當機。
3. ⏱️ **Google 平台搶佔 / 逾時 (Preemption / Idle Timeout)**：免費版 T4 GPU 在閒置或長時間執行時會被 Google 伺服器動態回收。

**本筆記本的全方位防禦與回溯機制：**
1. 🛡️ **主動防崩潰優化**：鎖定 `num_workers=2` 降低記憶體壓力，啟用 `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` 防止顯存碎片。
2. 🛰️ **黑盒子飛行記錄器 (Telemetry Daemon)**：背景程序每 30 秒自動向 Google Drive 記錄 RAM%、VRAM、磁碟、目前步數、Loss、OOM 訊號。
3. 🩺 **事後回溯診斷器 (Post-Mortem Analyzer)**：中斷後重新連線，執行步驟 7 即可一鍵查明最後崩潰原因與硬體趨勢！
4. 💾 **原子化雲端同步**：每次存檔以原子替換方式同步至 Google Drive，確保雲端永遠有一份完整可用的最新 Checkpoint。
5. 🔁 **真正斷點續訓 (`--resume=true`)**：重新連線後按「全部執行」，自動從同一步數、同一學習率排程接續！

⚠️ 請先確認硬體：**執行階段 (Runtime) -> 變更執行階段類型 (Change runtime type) -> 選擇 T4 GPU**

In [ ]:
# 1. 驗證 GPU 並取得 LeRobot 程式碼
import os, torch

if not torch.cuda.is_available():
    raise SystemError("❌ 未檢測到 GPU！請至 [執行階段 -> 變更執行階段類型] 切換至 T4 GPU 後再執行。")
print("✅ GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

if not os.path.exists("/content/lerobot"):
    os.system("git clone https://github.com/Child-pi/lerobot.git /content/lerobot")
else:
    os.system("cd /content/lerobot && git pull origin main")
%cd /content/lerobot

### ⚙️ 步驟 2：訓練設定與 Google Drive 掛載
- `TOTAL_STEPS`：**整個訓練的目標總步數**（接續訓練時可設為 10000 ~ 20000 步）。
- `SAVE_FREQ`：存檔與雲端同步頻率（建議 500 ~ 1000 步）。
- `AUTO_RESUME`：勾選時自動從 Drive 上次 Checkpoint 接續訓練。

In [ ]:
#@title 訓練設定 (Training Settings)
TOTAL_STEPS = 20000 #@param {type:"integer"}
SAVE_FREQ = 500 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
AUTO_RESUME = True #@param {type:"boolean"}
DATASET_REPO_ID = "lerobot/aloha_sim_insertion_human" #@param ["lerobot/aloha_sim_insertion_human", "lerobot/aloha_sim_transfer_cube_human"]
DRIVE_ROOT = "/content/drive/MyDrive/SmolVLA_training" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# 與舊版筆記本相容路徑對接
os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/SmolVLA_training"):
    try:
        os.symlink(DRIVE_ROOT, "/content/data/SmolVLA_training")
    except OSError:
        pass

LEROBOT_DIR = "/content/lerobot"
OUTPUT_DIR = os.path.join(LEROBOT_DIR, "outputs/train/smolvla_aloha")
LOCAL_CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
DRIVE_CKPT = os.path.join(DRIVE_ROOT, "checkpoint_latest")   # 最新完整 Checkpoint
LOG_FILE = "/content/train_log.txt"                          # 本機日誌（即時讀寫）
DRIVE_LOG_FILE = os.path.join(DRIVE_ROOT, "train_log.txt")   # 雲端同步日誌
TELEMETRY_CSV = os.path.join(DRIVE_ROOT, "telemetry_flight_recorder.csv") # 黑盒子歷史記錄
HEARTBEAT_JSON = os.path.join(DRIVE_ROOT, "last_heartbeat.json")          # 最新心跳狀態
PID_FILE = "/content/train.pid"

def train_running():
    # 檢查訓練進程是否存在且非 Zombie
    try:
        pid = int(open(PID_FILE).read().strip())
        with open(f"/proc/{pid}/status") as fh:
            for line in fh:
                if line.startswith("State:"):
                    return "Z" not in line and "X" not in line
        return True
    except Exception:
        return False

print("✅ Drive 已掛載:", DRIVE_ROOT)
print(f"🎯 目標總步數: {TOTAL_STEPS}，每 {SAVE_FREQ} 步存檔並同步到 Drive")
print("🛰️ 黑盒子飛行記錄將即時寫入:", TELEMETRY_CSV)

### 📦 步驟 3：安裝依賴（環境初始化）

In [ ]:
%cd /content/lerobot
!pip install -q -e ".[smolvla]"
!pip install -q dm_env dm_tree glfw lxml pyopengl pyparsing
!pip install -q --upgrade "mujoco>=3.15.0"
!pip install -q dm-control --no-deps
!pip install -q gym-aloha --no-deps
!pip install -q av pandas
print("🎉 環境安裝完成！")

### 🔁 步驟 4：從 Google Drive 還原最新 Checkpoint
- 若 Drive 上有完整的 `checkpoint_latest/` ➔ **真正續訓**（步數、optimizer、學習率完全無縫接續）。
- 若只有舊版權重 ➔ 自動做起點微調。
- 無存檔 ➔ 從 `lerobot/smolvla_base` 全新開始。

In [ ]:
import os, glob, shutil, json

RESUME_MODE = "fresh"
WARM_START_PATH = None
resume_step = 0

def _read_step(ckpt_dir):
    p = os.path.join(ckpt_dir, "training_state", "training_step.json")
    if os.path.exists(p):
        with open(p) as f:
            return int(json.load(f).get("step", 0))
    return int(os.path.basename(ckpt_dir)) if os.path.basename(ckpt_dir).isdigit() else 0

step_file = os.path.join(DRIVE_CKPT, "STEP")
if AUTO_RESUME and os.path.exists(step_file):
    step_name = open(step_file).read().strip()
    src = os.path.join(DRIVE_CKPT, step_name)
    dst = os.path.join(LOCAL_CKPT_DIR, step_name)
    if os.path.isdir(src):
        if os.path.exists(OUTPUT_DIR):
            shutil.rmtree(OUTPUT_DIR)
        print(f"⏬ 從 Drive 還原 Checkpoint {step_name}...")
        shutil.copytree(src, dst)
        os.symlink(step_name, os.path.join(LOCAL_CKPT_DIR, "last"))
        resume_step = _read_step(dst)
        RESUME_MODE = "resume"
        print(f"✅ 將從第 {resume_step} 步真正接續訓練（目標總步數: {TOTAL_STEPS}）")
elif AUTO_RESUME:
    legacy = sorted(glob.glob(os.path.join(DRIVE_ROOT, "outputs/train/*/checkpoints/*/pretrained_model")),
                    key=os.path.getmtime)
    if legacy:
        WARM_START_PATH = "/content/warm_start_model"
        shutil.rmtree(WARM_START_PATH, ignore_errors=True)
        shutil.copytree(legacy[-1], WARM_START_PATH)
        RESUME_MODE = "warm_start"
        print("♻️ 找到舊版權重，作為起點微調:", legacy[-1])

if RESUME_MODE == "fresh":
    print("🆕 從 lerobot/smolvla_base 全新開始訓練")
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)

### 🚀 步驟 5：背景啟動訓練 + 黑盒子飛行記錄器常駐
此儲存格會**立即結束**。訓練與黑盒子監控程式在背景持續運作，隨時將進度與硬體健康狀態寫入 Google Drive。

In [ ]:
import os, subprocess, shlex, time

DAEMON_SCRIPT = "/content/telemetry_and_sync_daemon.py"
with open(DAEMON_SCRIPT, "w", encoding="utf-8") as f:
    f.write('''# -*- coding: utf-8 -*-
import os, sys, time, shutil, subprocess, json, re, csv

local_dir, drive_dst, pid_file, log_file, drive_log, telemetry_csv, heartbeat_json = sys.argv[1:8]
synced = None

_PROG = re.compile(r"Training:\s*\d+%.*?\|\s*(\d+)/(\d+)")

def get_train_status(pid):
    try:
        with open(f"/proc/{pid}/status") as fh:
            lines = fh.readlines()
        state = "UNKNOWN"
        rss_mb = 0.0
        for l in lines:
            if l.startswith("State:"):
                state = l.split(":")[1].strip()
            elif l.startswith("VmRSS:"):
                rss_mb = round(int(l.split(":")[1].strip().split()[0]) / 1024.0, 1)
        alive = ("Z" not in state and "X" not in state)
        return alive, state, rss_mb
    except Exception:
        return False, "DEAD", 0.0

def get_sys_health():
    # RAM
    ram_used_gb, ram_total_gb, ram_pct = 0.0, 0.0, 0.0
    if os.path.exists("/proc/meminfo"):
        mem = {}
        for line in open("/proc/meminfo"):
            p = line.split(":")
            if len(p) == 2:
                mem[p[0].strip()] = int(p[1].split()[0])
        total = mem.get("MemTotal", 1)
        avail = mem.get("MemAvailable", mem.get("MemFree", 0))
        used = total - avail
        ram_used_gb = round(used / 1024**2, 2)
        ram_total_gb = round(total / 1024**2, 2)
        ram_pct = round(used / total * 100, 1)

    # Disk
    du = shutil.disk_usage("/content")
    disk_used_gb = round((du.total - du.free) / 1024**3, 2)
    disk_free_gb = round(du.free / 1024**3, 2)
    disk_pct = round((du.total - du.free) / du.total * 100, 1)

    # OOM Kill counter
    oom_count = 0
    if os.path.exists("/proc/vmstat"):
        for line in open("/proc/vmstat"):
            if line.startswith("oom_kill"):
                oom_count = int(line.split()[1])

    # GPU
    vram_used_mb, vram_total_mb, gpu_util_pct, gpu_temp_c = 0.0, 0.0, 0.0, 0.0
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu", "--format=csv,nounits,noheader"],
            text=True, stderr=subprocess.DEVNULL
        ).strip().split(",")
        vram_used_mb = float(out[0].strip())
        vram_total_mb = float(out[1].strip())
        gpu_util_pct = float(out[2].strip())
        gpu_temp_c = float(out[3].strip())
    except Exception:
        pass

    return {
        "ram_used_gb": ram_used_gb, "ram_total_gb": ram_total_gb, "ram_pct": ram_pct,
        "disk_used_gb": disk_used_gb, "disk_free_gb": disk_free_gb, "disk_pct": disk_pct,
        "vram_used_mb": vram_used_mb, "vram_total_mb": vram_total_mb,
        "gpu_util_pct": gpu_util_pct, "gpu_temp_c": gpu_temp_c,
        "oom_count": oom_count
    }

def get_train_progress():
    step, total_steps, loss = None, None, None
    if os.path.exists(log_file):
        try:
            with open(log_file, "r", errors="ignore") as f:
                f.seek(max(0, os.path.getsize(log_file) - 30720))
                text = f.read()
                matches = list(_PROG.finditer(text))
                if matches:
                    step = int(matches[-1].group(1))
                    total_steps = int(matches[-1].group(2))
                loss_matches = re.findall(r"loss:([0-9\.]+)", text)
                if loss_matches:
                    loss = float(loss_matches[-1])
        except Exception:
            pass
    return step, total_steps, loss

def sync_checkpoint_if_any():
    global synced
    link = os.path.join(local_dir, "last")
    if not os.path.islink(link):
        return
    target = os.path.realpath(link)
    step = os.path.basename(target)
    if step == synced or not os.path.isdir(target):
        return
    time.sleep(5)
    tmp, old = drive_dst + ".tmp", drive_dst + ".old"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(target, os.path.join(tmp, step))
    with open(os.path.join(tmp, "STEP"), "w") as fh:
        fh.write(step)
    shutil.rmtree(old, ignore_errors=True)
    if os.path.exists(drive_dst):
        os.rename(drive_dst, old)
    os.rename(tmp, drive_dst)
    shutil.rmtree(old, ignore_errors=True)
    synced = step
    print(time.strftime("%H:%M:%S"), "☁️ [Sync] Checkpoint", step, "已安全同步至 Google Drive", flush=True)
    for d in os.listdir(local_dir):
        p = os.path.join(local_dir, d)
        if d not in ("last", step) and os.path.isdir(p) and not os.path.islink(p):
            shutil.rmtree(p, ignore_errors=True)

# 初始檢查
init_oom = 0
try:
    if os.path.exists("/proc/vmstat"):
        for l in open("/proc/vmstat"):
            if l.startswith("oom_kill"): init_oom = int(l.split()[1])
except Exception: pass

pid = int(open(pid_file).read().strip())
csv_header_written = os.path.exists(telemetry_csv)

print("🛰️ [Telemetry] 黑盒子飛行記錄器啟動成功，每 30 秒向 Google Drive 回傳健康心跳...", flush=True)

while True:
    alive, proc_state, proc_rss_mb = get_train_status(pid)
    sys_h = get_sys_health()
    step, total_steps, loss = get_train_progress()

    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "alive": alive,
        "proc_state": proc_state,
        "proc_rss_mb": proc_rss_mb,
        **sys_h,
        "step": step or 0,
        "total_steps": total_steps or 0,
        "loss": loss or 0.0,
        "status": "RUNNING" if alive else "STOPPED"
    }

    # 1. 寫入 CSV 飛行記錄表
    try:
        write_header = not os.path.exists(telemetry_csv) or os.path.getsize(telemetry_csv) == 0
        with open(telemetry_csv, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(record.keys()))
            if write_header:
                w.writeheader()
            w.writerow(record)
    except Exception as e:
        print("⚠️ 寫入 CSV 失敗:", e, flush=True)

    # 2. 寫入 JSON 心跳狀態
    try:
        hb_tmp = heartbeat_json + ".tmp"
        with open(hb_tmp, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        os.replace(hb_tmp, heartbeat_json)
    except Exception: pass

    # 3. 備份 Log
    try:
        shutil.copyfile(log_file, drive_log)
    except Exception: pass

    # 4. 同步 Checkpoint
    try:
        sync_checkpoint_if_any()
    except Exception as e:
        print("⚠️ Checkpoint 同步失敗:", e, flush=True)

    if not alive:
        # 進程死亡，進行事後判定
        diag = "NORMAL_EXIT"
        if sys_h["oom_count"] > init_oom or sys_h["ram_pct"] >= 96.0:
            diag = "🛑 CRASH: 系統記憶體 (RAM) 耗盡，遭 Linux 核心 OOM-Killer 強制終止 (SIGKILL)！"
        elif sys_h["disk_free_gb"] < 1.5:
            diag = "🛑 CRASH: 本機磁碟空間不足 (<1.5GB) 造成系統崩潰！"
        elif step and total_steps and step >= total_steps:
            diag = "🟢 SUCCESS: 訓練已順利抵達目標總步數！"
        else:
            diag = "⚠️ UNEXPECTED_EXIT: 程序無預警中斷（硬體正常，可能為 Google Colab 搶佔回收或瀏覽器逾時切斷）。"
        record["final_diagnosis"] = diag
        record["alive"] = False
        try:
            with open(heartbeat_json, "w", encoding="utf-8") as f:
                json.dump(record, f, indent=2, ensure_ascii=False)
        except Exception: pass
        print("🛰️ [Telemetry] 檢測到訓練進程結束。事故判定:", diag, flush=True)
        break

    time.sleep(30)

print("✅ 黑盒子守護程式任務完成並結束。", flush=True)
''')

if train_running():
    print("ℹ️ 訓練已經在背景執行中，請直接執行步驟 6 觀察進度。")
elif RESUME_MODE == "resume" and resume_step >= TOTAL_STEPS:
    print(f"🏁 已完成 {resume_step} 步 ≥ TOTAL_STEPS={TOTAL_STEPS}。若要再訓練，請在步驟 2 調高 TOTAL_STEPS。")
else:
    # 主動防崩潰參數：num_workers=2 降低 RAM 壓力，避免 OOM-Killer
    common = [
        f"--steps={TOTAL_STEPS}",
        f"--save_freq={SAVE_FREQ}",
        f"--batch_size={BATCH_SIZE}",
        f"--output_dir={OUTPUT_DIR}",
        "--num_workers=2",
        "--log_freq=20",
        "--wandb.enable=false",
        "--policy.push_to_hub=false",
    ]
    if RESUME_MODE == "resume":
        cmd = ["lerobot-train",
               f"--config_path={os.path.join(LOCAL_CKPT_DIR, 'last/pretrained_model/train_config.json')}",
               "--resume=true"] + common
    else:
        cmd = ["lerobot-train",
               f"--policy.path={WARM_START_PATH or 'lerobot/smolvla_base'}",
               f"--dataset.repo_id={DATASET_REPO_ID}",
               "--policy.empty_cameras=2",
               '--rename_map={"observation.images.top": "observation.images.camera1"}'] + common

    # 啟用 CUDA 記憶體碎片整理與多線程優化
    shell_cmd = (
        f"cd {LEROBOT_DIR} && "
        f"MUJOCO_GL=egl PYTHONUNBUFFERED=1 "
        f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=2 "
        f"nohup {' '.join(shlex.quote(c) for c in cmd)} >> {shlex.quote(LOG_FILE)} 2>&1 & echo $! > {PID_FILE}"
    )
    with open(LOG_FILE, "a") as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} 啟動 ({RESUME_MODE}) =====\n")
    subprocess.run(shell_cmd, shell=True, check=True)
    time.sleep(3)
    subprocess.run(
        f"nohup python {DAEMON_SCRIPT} {shlex.quote(LOCAL_CKPT_DIR)} {shlex.quote(DRIVE_CKPT)} {PID_FILE} "
        f"{shlex.quote(LOG_FILE)} {shlex.quote(DRIVE_LOG_FILE)} {shlex.quote(TELEMETRY_CSV)} {shlex.quote(HEARTBEAT_JSON)} "
        f">> /content/sync_log.txt 2>&1 &",
        shell=True, check=True)
    print("🚀 訓練已在背景啟動！PID =", open(PID_FILE).read().strip())
    print("🛰️ 黑盒子飛行記錄器同步啟動中...")
    print("📄 日誌位置:", LOG_FILE)
    print("👉 請執行步驟 6 觀察即時進度（可隨時按停止，訓練不會中斷）")

### 📈 步驟 6：觀察訓練進度（會自動結束）
- 訓練會跑數個小時，這格**不需要一直等到訓練結束**：預設觀察 `MONITOR_MINUTES` 分鐘後自動結束，**訓練仍在背景繼續**。
- 隨時按 ■ 停止此格也不會影響訓練；想再看進度就重新執行這一格。

In [ ]:
#@title 觀察訓練進度
MONITOR_MINUTES = 10 #@param {type:"integer"}
import os, time, re

_PROG = re.compile(r"Training:\s*\d+%.*?\|\s*(\d+)/(\d+)\s*\[([^\]<]*)<([^\],]*)")

def _status(text):
    m = None
    for m in _PROG.finditer(text):
        pass
    if m:
        done, total, elapsed, remain = int(m.group(1)), int(m.group(2)), m.group(3).strip(), m.group(4).strip()
        return f"進度 {done}/{total} ({done / max(total, 1) * 100:.1f}%) | 已用 {elapsed} | 預估剩餘 {remain}"
    return "啟動中（載入資料集 / 模型，首次約需數分鐘）"

if not os.path.exists(LOG_FILE):
    print("⚠️ 找不到日誌，請先執行步驟 5 啟動訓練。")
else:
    pos = max(0, os.path.getsize(LOG_FILE) - 3000)
    deadline = time.time() + MONITOR_MINUTES * 60
    last_status_t = 0
    tail = ""
    try:
        while True:
            with open(LOG_FILE, errors="ignore") as f:
                f.seek(pos)
                chunk = f.read()
                pos = f.tell()
            tail = (tail + chunk)[-20000:]
            for line in re.split(r"[\r\n]+", chunk):
                s = line.strip()
                if s and "Training:" not in s and "check_worker_number_rationality" not in s \
                        and "Detected no triton" not in s:
                    print(s, flush=True)

            running = train_running()
            if time.time() - last_status_t >= 60 or not running:
                print(f"⏳ [{time.strftime('%H:%M:%S')}] {_status(tail)}", flush=True)
                last_status_t = time.time()

            if not running:
                if "Traceback" in tail or "Error" in tail.split("啟動")[-1][-3000:]:
                    print("\n❌ 訓練程序已結束，日誌中有錯誤訊息，請查看上方內容。")
                else:
                    print("\n🏁 訓練程序已結束。執行步驟 7 可查看詳細事故回溯，或執行步驟 8 進行評估。")
                break
            if time.time() >= deadline:
                print(f"\n✅ 已觀察 {MONITOR_MINUTES} 分鐘，此格自動結束；訓練仍在背景執行中。")
                print("👉 想再看進度，重新執行這一格即可。日誌已同步到:", DRIVE_LOG_FILE)
                break
            time.sleep(10)
    except KeyboardInterrupt:
        print("\n👋 已停止觀察；訓練仍在背景執行。")

### 🩺 步驟 7：黑盒子事故回溯診斷器 (Post-Mortem Health & Crash Analyzer)
**中斷離線、訓練結束或遇到問題時執行此格！**
自動從 Google 雲端硬碟讀取黑盒子飛行記錄，分析最後關鍵指標與死因判定。

In [ ]:
#@title 執行黑盒子事後診斷 (Analyze Flight Recorder Telemetry)
import os, json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

print("=" * 70)
print("🛰️ 【SmolVLA 訓練黑盒子健康診斷報告】")
print("=" * 70)

if os.path.exists(HEARTBEAT_JSON):
    try:
        with open(HEARTBEAT_JSON, "r", encoding="utf-8") as f:
            hb = json.load(f)
        alive_badge = "🟢 正常執行中 (Running)" if hb.get("alive") else "🔴 已停止 (Stopped)"
        print(f"⏰ 最後記錄時間 : {hb.get('timestamp')}")
        print(f"📡 進程狀態     : {alive_badge} | 狀態碼: {hb.get('proc_state', 'N/A')} | 進程 RSS: {hb.get('proc_rss_mb', 0)} MB")
        print(f"🎯 訓練進度     : Step {hb.get('step', 0)} / {hb.get('total_steps', 0)} | Loss: {hb.get('loss', 0.0)}")
        print(f"🧠 系統 RAM     : {hb.get('ram_used_gb', 0)} / {hb.get('ram_total_gb', 0)} GB ({hb.get('ram_pct', 0)}%)")
        print(f"⚡ 顯卡顯存 VRAM : {hb.get('vram_used_mb', 0)} / {hb.get('vram_total_mb', 0)} MB | 使用率: {hb.get('gpu_util_pct', 0)}% | 溫度: {hb.get('gpu_temp_c', 0)}°C")
        print(f"💾 磁碟剩餘空間 : {hb.get('disk_free_gb', 0)} GB / 總使用率: {hb.get('disk_pct', 0)}%")
        print(f"⚠️ OOM Kill 次數 : {hb.get('oom_count', 0)}")
        print("-" * 70)
        
        diag = hb.get("final_diagnosis", "進程仍在正常運行中。")
        print(f"🔍 【事故死因判定】:\n👉 {diag}")
    except Exception as e:
        print("讀取心跳檔案錯誤:", e)
else:
    print("ℹ️ 尚未在雲端硬碟找到心跳檔案 (last_heartbeat.json)，請先執行步驟 5。")

if os.path.exists(TELEMETRY_CSV):
    print("\n" + "=" * 70)
    print("📊 【最近 15 筆飛行記錄（崩潰前趨勢）】:")
    print("=" * 70)
    try:
        df = pd.read_csv(TELEMETRY_CSV)
        cols = ["timestamp", "alive", "ram_pct", "vram_used_mb", "disk_free_gb", "step", "loss"]
        available_cols = [c for c in cols if c in df.columns]
        display(df[available_cols].tail(15))
    except Exception as e:
        print("讀取 CSV 報表提示:", e)
else:
    print("ℹ️ 尚未在雲端硬碟找到飛行記錄表 (telemetry_flight_recorder.csv)。")

### 🎬 步驟 8：模擬評估並錄製影片（訓練完成後執行）

In [ ]:
import os, glob, subprocess

ckpt = os.path.join(LOCAL_CKPT_DIR, "last", "pretrained_model")
if not os.path.exists(ckpt) and os.path.exists(os.path.join(DRIVE_CKPT, "STEP")):
    step_name = open(os.path.join(DRIVE_CKPT, "STEP")).read().strip()
    ckpt = os.path.join(DRIVE_CKPT, step_name, "pretrained_model")

if not os.path.exists(ckpt):
    print("❌ 找不到可評估的 Checkpoint，請確認訓練是否已有存檔。")
else:
    print("✅ 評估模型:", ckpt)
    eval_cmd = [
        "lerobot-eval",
        f"--policy.path={ckpt}",
        "--env.type=aloha",
        "--env.task=" + ("AlohaTransferCube-v0" if "transfer_cube" in DATASET_REPO_ID else "AlohaInsertion-v0"),
        "--eval.n_episodes=3",
        "--eval.batch_size=1",
        "--eval.use_async_envs=false",
        '--rename_map={"observation.images.top": "observation.images.camera1"}',
        "--output_dir=/content/lerobot/outputs/eval/smolvla_aloha",
    ]
    env = dict(os.environ, MUJOCO_GL="egl", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(eval_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(p.stdout.readline, ""):
        if "Running rollout" not in line:
            print(line, end="", flush=True)
    p.wait()
    print("✅ 評估完成" if p.returncode == 0 else f"⚠️ 評估結束，狀態碼 {p.returncode}")

In [ ]:
# 📺 播放影片並備份評估結果到 Drive
import glob, os, shutil
from IPython.display import Video, display

videos = sorted(glob.glob("/content/lerobot/outputs/eval/**/*.mp4", recursive=True), key=os.path.getmtime)
if videos:
    print("🎥", videos[-1])
    display(Video(videos[-1], embed=True, width=640))
    shutil.copytree("/content/lerobot/outputs/eval", os.path.join(DRIVE_ROOT, "eval"), dirs_exist_ok=True)
    print("☁️ 評估影片已備份到", os.path.join(DRIVE_ROOT, "eval"))
else:
    print("⚠️ 尚未找到影片")

### 🛑 停止訓練（需要時才執行）
停止後 Drive 上仍保留最後一次同步的 Checkpoint 與黑盒子記錄，之後可再「全部執行」接續訓練。

In [ ]:
import os, signal
try:
    pid = int(open(PID_FILE).read().strip())
    os.kill(pid, signal.SIGTERM)
    print("🛑 已送出停止訊號給訓練程序 PID", pid)
except Exception as e:
    print("ℹ️ 沒有正在執行的訓練:", e)

### 💤 步驟 9：任務完成自動關機斷線（省錢省點數，可選）

如果您使用的是 **Colab Pro / 付費算力（Compute Units）**，放著跑去睡覺時，可以執行此儲存格：
* 它會**等待訓練完成並確保 Google Drive 備份完畢**後，自動中斷連線並釋放 GPU，**立刻停止計費**！

In [ ]:
#@title 💤 自動關機斷線 (Auto-Shutdown & Disconnect to Save Compute Units)
#@markdown 勾選下方選項後，此儲存格會等待訓練結束、檔案同步完畢後，自動終止 Colab 釋放 GPU，避免白白扣除運算點數。
WAIT_FOR_TRAIN_FINISH = True #@param {type:"boolean"}

import os, time

def _is_training_active():
    try:
        if os.path.exists("/content/lerobot_train.pid"):
            with open("/content/lerobot_train.pid", "r") as f:
                pid = int(f.read().strip())
            os.kill(pid, 0)
            return True
    except Exception:
        pass
    return False

if WAIT_FOR_TRAIN_FINISH and _is_training_active():
    print("⏳ 偵測到背景訓練進行中，正在等待訓練完成...")
    print("💡 您可以安心去休息，訓練結束後本儲存格會自動終止 Colab 關機！")
    while _is_training_active():
        time.sleep(30)
    print("🎉 訓練已結束！等待 15 秒確保飛行記錄器與權重完整同步至 Google Drive...")
    time.sleep(15)

print("🔌 正在中斷連線並釋放 GPU 資源，立刻停止計費...")
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"⚠️ 無法自動中斷連線: {e}，請至上方選單手動點選「執行階段」->「中斷連線並刪除執行階段」")
